<a href="https://colab.research.google.com/github/krabishri007/MPLADS_SIH/blob/main/MPLADS_AI_Risk_Intelligence_Engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q scikit-learn pandas numpy

In [2]:
import pandas as pd
import numpy as np
import re
import json
from datetime import datetime, timezone

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

pd.set_option('display.max_colwidth', 120)
print("Libraries loaded ✅")


Libraries loaded ✅


In [3]:
from google.colab import files

print("Upload the 4 MPLADS CSV files (you can select all 4 at once):")
uploaded = files.upload()   # opens a file picker in the Colab UI


Upload the 4 MPLADS CSV files (you can select all 4 at once):


Saving mplads_completed_works_2026-09-13.csv to mplads_completed_works_2026-09-13.csv
Saving mplads_expenditures_2026-09-10.csv to mplads_expenditures_2026-09-10.csv
Saving mplads_mp_summary_2026-09-10.csv to mplads_mp_summary_2026-09-10.csv
Saving mplads_recommended_works_2026-09-10.csv to mplads_recommended_works_2026-09-10.csv


In [4]:
# --- Alternative: mount Google Drive instead of uploading each time ---
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_DIR = '/content/drive/MyDrive/mplads_data'   # <- change to your folder
# Then skip the files.upload() cell above.

import glob

def find_file(keyword):
    """Finds the uploaded file whose name contains `keyword`, case-insensitive."""
    matches = [f for f in uploaded.keys() if keyword.lower() in f.lower()]
    if not matches:
        raise FileNotFoundError(f"No uploaded file matched '{keyword}'. "
                                 f"Files uploaded: {list(uploaded.keys())}")
    return matches[0]

completed_path   = find_file('completed')
recommended_path = find_file('recommended')
expenditure_path = find_file('expenditure')
mp_summary_path  = find_file('mp_summary')

print("Matched files:")
print(" completed  :", completed_path)
print(" recommended:", recommended_path)
print(" expenditure:", expenditure_path)
print(" mp_summary :", mp_summary_path)


Matched files:
 completed  : mplads_completed_works_2026-09-13.csv
 recommended: mplads_recommended_works_2026-09-10.csv
 expenditure: mplads_expenditures_2026-09-10.csv
 mp_summary : mplads_mp_summary_2026-09-10.csv


In [5]:
comp = pd.read_csv(completed_path)
rec  = pd.read_csv(recommended_path)
exp  = pd.read_csv(expenditure_path)
mps  = pd.read_csv(mp_summary_path)

# Strip stray whitespace from column headers (CSV exports sometimes have this)
for df in (comp, rec, exp, mps):
    df.columns = [c.strip() for c in df.columns]

print("Completed works :", comp.shape)
print("Recommended works:", rec.shape)
print("Expenditures     :", exp.shape)
print("MP summary       :", mps.shape)
comp.head(3)


Completed works : (44028, 12)
Recommended works: (87272, 11)
Expenditures     : (108695, 10)
MP summary       : (774, 16)


,Work ID,Work Description,Category,MP Name,Constituency,State,House,Final Amount (₹),Completed Date,Has Images,Average Rating,IDA
0,134703,Upgradation of Road from Madhavaram Village to Company Indlu of Madhavaram GP,Normal/Others,DAGGUMALLA PRASADA RAO,CHITTOOR,Andhra Pradesh,Lok Sabha,499993.0,2025-01-31T00:00:00.000Z,True,NaN,CHITTOOR(DISTRICT COLLECTOR CHITTOOR_IDA)
1,135593,Construction of CC Road from Amudala Village to AmudalaHW in Amudala Village&GP of Palasamudram MAndal of GD Nellore...,Normal/Others,DAGGUMALLA PRASADA RAO,CHITTOOR,Andhra Pradesh,Lok Sabha,448722.0,2024-12-05T00:00:00.000Z,True,NaN,CHITTOOR(DISTRICT COLLECTOR CHITTOOR_IDA)
2,135595,Construction of CC road from Amudala Village to Amudala ST colony in Amudala Village & GP of Palasamudram Mandal of ...,Normal/Others,DAGGUMALLA PRASADA RAO,CHITTOOR,Andhra Pradesh,Lok Sabha,448970.0,2024-12-05T00:00:00.000Z,True,NaN,CHITTOOR(DISTRICT COLLECTOR CHITTOOR_IDA)


In [6]:
WORK_TYPE_KEYWORDS = {
    'road_transport'     : r'\b(road|street|pathway|bridge|culvert|footpath|highway)\b',
    'drainage'           : r'\b(drain|drainage|sewer|sewerage)\b',
    'drinking_water'     : r'\b(drinking water|water supply|bore ?well|hand pump|overhead tank|pipeline)\b',
    'sanitation'         : r'\b(toilet|sanitation|washroom|urinal)\b',
    'education'          : r'\b(school|vidyalaya|classroom|anganwadi|college|library)\b',
    'health'             : r'\b(hospital|health.?center|health.?centre|phc|dispensary|ambulance|medical)\b',
    'community_hall'     : r'\b(community (center|centre|hall)|panchayat bhawan|marriage hall|kalyan mandap|assembly hall)\b',
    'solar_lighting'     : r'\b(solar|street light|led light|high mast)\b',
    'sports'             : r'\b(playground|stadium|sports|gym)\b',
    'furniture_equipment': r'\b(furniture|desk|bench|chair|table|tricycle|wheelchair|equipment)\b',
    'electricity'        : r'\b(electric|transformer|substation)\b',
    'crematorium'        : r'\b(crematorium|shamshan|graveyard|cemetery)\b',
    'agriculture'        : r'\b(irrigation|canal|agricultur|farm)\b',
}

def classify_work_type(description: str) -> str:
    """Returns the first matching work-type keyword bucket, else 'other'."""
    text = str(description).lower()
    for label, pattern in WORK_TYPE_KEYWORDS.items():
        if re.search(pattern, text):
            return label
    return 'other'


def normalize_description(description: str) -> str:
    """Strips directional/serial words (NORTH/SOUTH/PART/PHASE/numbers) so that
    'X North Part' and 'X South Part' collapse to the same normalized text.
    This is what lets us later detect a single work artificially split into
    many near-identical smaller works."""
    text = str(description).lower()
    text = re.sub(r'\b(north|south|east|west|part|phase|reach|zone|block|sector|no\.?\s*\d+|\d+)\b', ' ', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print("Classifier ready ✅")


Classifier ready ✅


In [7]:
comp2 = comp.rename(columns={'Final Amount (₹)': 'amount', 'Completed Date': 'date'}).copy()
comp2['status'] = 'Completed'
comp2['has_images'] = comp2['Has Images'].fillna(False)

rec2 = rec.rename(columns={'Recommended Amount (₹)': 'amount', 'Recommendation Date': 'date'}).copy()
rec2['status'] = 'Recommended (pending)'
rec2['has_images'] = rec2['Has Images'].fillna(False)

KEEP_COLS = ['Work ID', 'Work Description', 'MP Name', 'Constituency', 'State',
             'House', 'amount', 'date', 'has_images', 'status', 'IDA']

works = pd.concat([comp2[KEEP_COLS], rec2[KEEP_COLS]], ignore_index=True)

# Clean amounts
works['amount'] = pd.to_numeric(works['amount'], errors='coerce')
works = works.dropna(subset=['amount'])
works = works[works['amount'] > 0].reset_index(drop=True)

# Apply the classifiers from Section 2
works['work_type'] = works['Work Description'].apply(classify_work_type)
works['norm_desc']  = works['Work Description'].apply(normalize_description)
works['log_amount'] = np.log1p(works['amount'])

print("Unified works table:", works.shape)
works[['Work Description', 'work_type', 'amount', 'status']].sample(5, random_state=1)


Unified works table: (131300, 14)


,Work Description,work_type,amount,status
15269,"Construction of PCC flooring at Vaisamtlang , Lengpui venghlui",other,1000000.0,Completed
73702,"Drilling of Borewell and Supply and Delivery of Pump set at Gemiya Naik Tanda,Cherla Tanda Hab of Gotlachelka Tanda...",drinking_water,200000.0,Recommended (pending)
32201,"Construction of CC Road from Ejahar sekh house to kochan sk house at Simulia-1 GP ,Executing Agency will be EO Monga...",road_transport,979687.0,Completed
49060,Construction of Interlocking Road from Balaji Jatka Mandir to Bijli Power House at Shimla Village,road_transport,1000000.0,Recommended (pending)
55014,Purchase of books P.P. Mukundan Smaraka Vayanasala and grandalayam,other,200000.0,Recommended (pending)


In [8]:
# --- 4a. Cost outlier: robust z-score of log(amount) within each work_type ---
grp = works.groupby('work_type')['log_amount']
median = grp.transform('median')
q1 = grp.transform(lambda s: s.quantile(0.25))
q3 = grp.transform(lambda s: s.quantile(0.75))
iqr = (q3 - q1).replace(0, np.nan)

works['cost_robust_z'] = ((works['log_amount'] - median) / iqr).fillna(0)
works['flag_cost_outlier'] = works['cost_robust_z'].abs() > 2.5


In [9]:
# --- 4b. Suspiciously round amounts (₹1,00,000 multiples, above ₹1 lakh) ---
works['flag_round_amount'] = (works['amount'] % 100_000 == 0) & (works['amount'] >= 100_000)


In [10]:
# --- 4c. Missing evidence: only meaningful for COMPLETED works ---
works['flag_missing_evidence'] = (works['status'] == 'Completed') & (~works['has_images'])


In [11]:
# --- 4d. Fragmentation / possible work-splitting ---
# Group by (MP, normalized description, exact amount). If 5+ works share all
# three, they are near-identical works billed at an identical figure.
fragment_groups = (
    works.groupby(['MP Name', 'norm_desc', 'amount'])
         .size()
         .reset_index(name='cluster_size')
)
fragment_groups = fragment_groups[
    (fragment_groups['norm_desc'].str.len() > 3) & (fragment_groups['cluster_size'] >= 5)
]
cluster_lookup = fragment_groups.set_index(['MP Name', 'norm_desc', 'amount'])['cluster_size'].to_dict()

works['fragmentation_cluster_size'] = (
    works.set_index(['MP Name', 'norm_desc', 'amount']).index.map(cluster_lookup).fillna(0)
)
works['flag_fragmentation'] = works['fragmentation_cluster_size'] >= 5

print("Flag rates:")
for col in ['flag_cost_outlier', 'flag_round_amount', 'flag_missing_evidence', 'flag_fragmentation']:
    print(f"  {col:<24s}: {works[col].mean()*100:5.2f}%  ({works[col].sum()} works)")


Flag rates:
  flag_cost_outlier       :  1.49%  (1962 works)
  flag_round_amount       : 39.70%  (52124 works)
  flag_missing_evidence   :  9.71%  (12746 works)
  flag_fragmentation      :  7.45%  (9785 works)


In [12]:
RULE_WEIGHTS = {
    'flag_cost_outlier'    : 30,
    'flag_round_amount'    : 10,
    'flag_missing_evidence': 20,
    'flag_fragmentation'   : 40,   # weighted highest: pattern most associated with fund misuse
}

works['rule_score'] = sum(works[flag].astype(int) * weight for flag, weight in RULE_WEIGHTS.items())
works['rule_score'] = works['rule_score'].clip(0, 100)

works['rule_score'].describe()


,rule_score
count,131300.000000
mean,9.340594
std,12.691629
min,0.000000
25%,0.000000
50%,10.000000
75%,10.000000
max,90.000000


In [13]:
FEATURE_COLS = ['log_amount', 'cost_robust_z', 'fragmentation_cluster_size']

X = works[FEATURE_COLS].fillna(0).values
X_scaled = StandardScaler().fit_transform(X)

iso_forest = IsolationForest(
    n_estimators=200,
    contamination=0.08,   # assume ~8% of works look anomalous — tune this during the pilot
    random_state=42,
)
iso_forest.fit(X_scaled)

# score_samples: higher = more "normal". We flip the sign so higher = more anomalous.
raw_anomaly = -iso_forest.score_samples(X_scaled)
works['ml_anomaly_score'] = (
    (raw_anomaly - raw_anomaly.min()) / (raw_anomaly.max() - raw_anomaly.min()) * 100
)

works['ml_anomaly_score'].describe()


,ml_anomaly_score
count,131300.000000
mean,16.001596
std,18.918683
min,0.000000
25%,3.690011
50%,7.627376
75%,19.967070
max,100.000000


In [14]:
works['final_risk_score'] = (0.6 * works['rule_score'] + 0.4 * works['ml_anomaly_score']).round(1)

def risk_band(score):
    if score >= 60:
        return 'High'
    elif score >= 30:
        return 'Medium'
    return 'Low'

works['risk_band'] = works['final_risk_score'].apply(risk_band)

works['risk_band'].value_counts()


,count
risk_band,
Low,118622
Medium,11740
High,938


In [15]:
REASON_TEXT = {
    'flag_cost_outlier'    : "Cost is a statistical outlier vs similar '{wt}' works nationwide",
    'flag_round_amount'    : "Amount is an unusually round figure (₹{amt:,.0f})",
    'flag_missing_evidence': "Marked completed but has no supporting site photos uploaded",
    'flag_fragmentation'   : "Part of a cluster of {n} near-identical works by the same MP (possible work-splitting)",
}
VERIFY_TEXT = {
    'flag_cost_outlier'    : "Compare against the state Schedule of Rates (SOR) for this work category",
    'flag_round_amount'    : "Cross-check the vendor quotation/estimate backing this exact figure",
    'flag_missing_evidence': "Request geo-tagged site photos or schedule a physical inspection",
    'flag_fragmentation'   : "Confirm whether this is a legitimate sanctioned batch scheme, or works were split to avoid approval/tender thresholds",
}

def build_reasons(row):
    reasons, checklist = [], []
    if row['flag_cost_outlier']:
        reasons.append(REASON_TEXT['flag_cost_outlier'].format(wt=row['work_type'].replace('_', ' ')))
        checklist.append(VERIFY_TEXT['flag_cost_outlier'])
    if row['flag_round_amount']:
        reasons.append(REASON_TEXT['flag_round_amount'].format(amt=row['amount']))
        checklist.append(VERIFY_TEXT['flag_round_amount'])
    if row['flag_missing_evidence']:
        reasons.append(REASON_TEXT['flag_missing_evidence'])
        checklist.append(VERIFY_TEXT['flag_missing_evidence'])
    if row['flag_fragmentation']:
        reasons.append(REASON_TEXT['flag_fragmentation'].format(n=int(row['fragmentation_cluster_size'])))
        checklist.append(VERIFY_TEXT['flag_fragmentation'])
    if not reasons:
        reasons = ["No strong anomaly signals detected"]
        checklist = ["Routine monitoring is sufficient"]
    return pd.Series([reasons, checklist])

works[['reasons', 'verify_checklist']] = works.apply(build_reasons, axis=1)

# Peek at a flagged example
works[works['risk_band'] == 'High'][
    ['Work Description', 'MP Name', 'final_risk_score', 'reasons', 'verify_checklist']
].head(3)


,Work Description,MP Name,final_risk_score,reasons,verify_checklist
1673,HIGH MAST LIGHT KA ADHISTHAPAN - 1,Dileshwar Kamait,66.3,"[Marked completed but has no supporting site photos uploaded, Part of a cluster of 50 near-identical works by the sa...","[Request geo-tagged site photos or schedule a physical inspection, Confirm whether this is a legitimate sanctioned b..."
1674,HIGH MAST LIGHT KA ADHISTHAPAN,Dileshwar Kamait,66.3,"[Marked completed but has no supporting site photos uploaded, Part of a cluster of 50 near-identical works by the sa...","[Request geo-tagged site photos or schedule a physical inspection, Confirm whether this is a legitimate sanctioned b..."
1675,HIGH MAST LIGHT KA ADHISTHAPAN -1,Dileshwar Kamait,66.3,"[Marked completed but has no supporting site photos uploaded, Part of a cluster of 50 near-identical works by the sa...","[Request geo-tagged site photos or schedule a physical inspection, Confirm whether this is a legitimate sanctioned b..."


In [16]:
mps['utilization_completion_gap'] = mps['Utilization %'] - mps['Completion Rate %']
mps['pending_ratio'] = mps['Pending Payments'] / mps['Transaction Count'].replace(0, np.nan)
mps['balance_ratio'] = (
    mps['Balance Not Yet Paid to Vendors (₹)'] / mps['Allocated Amount (₹)'].replace(0, np.nan)
)

# Vendor concentration per MP (Herfindahl-Hirschman Index — higher = more concentrated)
vendor_amt = exp.groupby(['MP Name', 'Vendor'])['Expenditure Amount (₹)'].sum().reset_index()
mp_total = vendor_amt.groupby('MP Name')['Expenditure Amount (₹)'].transform('sum')
vendor_amt['share'] = vendor_amt['Expenditure Amount (₹)'] / mp_total

hhi = vendor_amt.groupby('MP Name')['share'].apply(lambda s: (s ** 2).sum()).reset_index(name='vendor_hhi')
top_vendor = (
    vendor_amt.sort_values('share', ascending=False)
    .drop_duplicates('MP Name')[['MP Name', 'Vendor', 'share']]
)
top_vendor.columns = ['MP Name', 'top_vendor', 'top_vendor_share']

mps = mps.merge(hhi, on='MP Name', how='left').merge(top_vendor, on='MP Name', how='left')
mps['vendor_hhi'] = mps['vendor_hhi'].fillna(0)
mps['top_vendor_share'] = mps['top_vendor_share'].fillna(0)

mps['flag_util_completion_mismatch'] = mps['utilization_completion_gap'] >= 60
mps['flag_vendor_concentration'] = mps['top_vendor_share'] >= 0.4
mps['flag_high_pending_balance'] = mps['balance_ratio'] >= 0.5

print("MP-level flag rates:")
for col in ['flag_util_completion_mismatch', 'flag_vendor_concentration', 'flag_high_pending_balance']:
    print(f"  {col:<32s}: {mps[col].mean()*100:5.2f}%  ({mps[col].sum()} MPs)")


MP-level flag rates:
  flag_util_completion_mismatch   : 19.64%  (152 MPs)
  flag_vendor_concentration       : 22.87%  (177 MPs)
  flag_high_pending_balance       : 18.35%  (142 MPs)


In [17]:
MP_WEIGHTS = {
    'flag_util_completion_mismatch': 45,
    'flag_vendor_concentration'    : 30,
    'flag_high_pending_balance'    : 25,
}
mps['mp_rule_score'] = sum(mps[flag].astype(int) * weight for flag, weight in MP_WEIGHTS.items())

MP_FEATURE_COLS = ['utilization_completion_gap', 'vendor_hhi', 'balance_ratio']
Xm = mps[MP_FEATURE_COLS].fillna(0).values
Xm_scaled = StandardScaler().fit_transform(Xm)

iso_forest_mp = IsolationForest(n_estimators=200, contamination=0.1, random_state=42)
iso_forest_mp.fit(Xm_scaled)
raw_mp_anomaly = -iso_forest_mp.score_samples(Xm_scaled)
mps['mp_ml_score'] = (raw_mp_anomaly - raw_mp_anomaly.min()) / (raw_mp_anomaly.max() - raw_mp_anomaly.min()) * 100

mps['mp_final_risk_score'] = (0.6 * mps['mp_rule_score'] + 0.4 * mps['mp_ml_score']).round(1)
mps['mp_risk_band'] = mps['mp_final_risk_score'].apply(risk_band)

mps['mp_risk_band'].value_counts()


,count
mp_risk_band,
Low,524
Medium,192
High,58


In [18]:
MP_REASON_TEXT = {
    'flag_util_completion_mismatch': "{u:.0f}% of funds recorded as utilised but only {c:.0f}% of works are physically completed",
    'flag_vendor_concentration'    : "{share:.0f}% of this MP's expenditure has gone to a single vendor ({vendor})",
    'flag_high_pending_balance'    : "₹{bal:,.0f} allocated but still not paid out to any vendor",
}
MP_VERIFY_TEXT = {
    'flag_util_completion_mismatch': "Field-verify physical progress of works marked 'in progress' against reported spend",
    'flag_vendor_concentration'    : "Check vendor empanelment process and look for related-party or single-bidder tenders",
    'flag_high_pending_balance'    : "Review why sanctioned works have not moved to payment stage",
}

def build_mp_reasons(row):
    reasons, checklist = [], []
    if row['flag_util_completion_mismatch']:
        reasons.append(MP_REASON_TEXT['flag_util_completion_mismatch'].format(
            u=row['Utilization %'], c=row['Completion Rate %']))
        checklist.append(MP_VERIFY_TEXT['flag_util_completion_mismatch'])
    if row['flag_vendor_concentration']:
        reasons.append(MP_REASON_TEXT['flag_vendor_concentration'].format(
            share=row['top_vendor_share'] * 100, vendor=row['top_vendor']))
        checklist.append(MP_VERIFY_TEXT['flag_vendor_concentration'])
    if row['flag_high_pending_balance']:
        reasons.append(MP_REASON_TEXT['flag_high_pending_balance'].format(
            bal=row['Balance Not Yet Paid to Vendors (₹)']))
        checklist.append(MP_VERIFY_TEXT['flag_high_pending_balance'])
    if not reasons:
        reasons = ["No strong anomaly signals detected"]
        checklist = ["Routine monitoring is sufficient"]
    return pd.Series([reasons, checklist])

mps[['reasons', 'verify_checklist']] = mps.apply(build_mp_reasons, axis=1)
mps[mps['mp_risk_band'] == 'High'][['MP Name', 'State', 'mp_final_risk_score', 'reasons']].head(3)


,MP Name,State,mp_final_risk_score,reasons
1,RAVINDRA DATTARAM WAIKAR,Maharashtra,66.7,"[100% of funds recorded as utilised but only 0% of works are physically completed, ₹93,200,878 allocated but still n..."
3,Smt. Sudha Murty (2024-30),Karnataka,98.8,"[100% of funds recorded as utilised but only 0% of works are physically completed, 93% of this MP's expenditure has ..."
8,Shri Bhartruhari Mahtab,Odisha,60.5,"[100% of funds recorded as utilised but only 18% of works are physically completed, ₹109,904,509 allocated but still..."


In [19]:
import os
os.makedirs('mplads_ai_outputs', exist_ok=True)

# Full scored tables, sorted riskiest-first
works_out = works[[
    'Work ID', 'Work Description', 'MP Name', 'Constituency', 'State', 'House',
    'work_type', 'amount', 'date', 'status', 'has_images',
    'final_risk_score', 'risk_band', 'reasons', 'verify_checklist',
    'flag_cost_outlier', 'flag_round_amount', 'flag_missing_evidence', 'flag_fragmentation',
]].sort_values('final_risk_score', ascending=False)
works_out['Work ID'] = works_out['Work ID'].astype(int)

mps_out = mps[[
    'MP Name', 'Constituency', 'State', 'House',
    'Allocated Amount (₹)', 'Total Expenditure (₹)', 'Utilization %', 'Completion Rate %',
    'Completed Works', 'Recommended Works', 'Balance Not Yet Paid to Vendors (₹)',
    'top_vendor', 'top_vendor_share', 'mp_final_risk_score', 'mp_risk_band',
    'reasons', 'verify_checklist',
    'flag_util_completion_mismatch', 'flag_vendor_concentration', 'flag_high_pending_balance',
]].sort_values('mp_final_risk_score', ascending=False)

# The dashboard's "priority queue" only needs the top N riskiest works —
# officials want a short actionable list, not all ~130k rows.
TOP_N = 3000
works_out.head(TOP_N).to_json(f'mplads_ai_outputs/risk_scores_works.json', orient='records')
mps_out.to_json(f'mplads_ai_outputs/risk_scores_mps.json', orient='records')

# Pre-aggregated numbers for charts, computed on the FULL scored set
state_agg = (
    works_out.groupby('State')
    .agg(total_works=('final_risk_score', 'size'),
         high_risk=('risk_band', lambda s: (s == 'High').sum()),
         avg_risk=('final_risk_score', 'mean'))
    .reset_index().sort_values('high_risk', ascending=False)
)
worktype_agg = (
    works_out.groupby('work_type')
    .agg(total_works=('final_risk_score', 'size'),
         high_risk=('risk_band', lambda s: (s == 'High').sum()),
         avg_risk=('final_risk_score', 'mean'))
    .reset_index().sort_values('high_risk', ascending=False)
)
state_agg.to_json('mplads_ai_outputs/aggregate_by_state.json', orient='records')
worktype_agg.to_json('mplads_ai_outputs/aggregate_by_worktype.json', orient='records')

meta = {
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "total_works_scored": int(len(works_out)),
    "total_mps_scored": int(len(mps_out)),
    "high_risk_works": int((works_out['risk_band'] == 'High').sum()),
    "medium_risk_works": int((works_out['risk_band'] == 'Medium').sum()),
    "high_risk_mps": int((mps_out['mp_risk_band'] == 'High').sum()),
    "priority_queue_size": int(min(TOP_N, len(works_out))),
}
with open('mplads_ai_outputs/meta_summary.json', 'w') as f:
    json.dump(meta, f, indent=2)

print(json.dumps(meta, indent=2))


{
  "generated_at": "2026-09-13T07:43:08.956803+00:00",
  "total_works_scored": 131300,
  "total_mps_scored": 774,
  "high_risk_works": 938,
  "medium_risk_works": 11740,
  "high_risk_mps": 58,
  "priority_queue_size": 3000
}


In [20]:
# Download everything as one zip so you can hand it straight to the Streamlit app
import shutil
shutil.make_archive('mplads_ai_outputs', 'zip', 'mplads_ai_outputs')
files.download('mplads_ai_outputs.zip')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [34]:
import requests
import time

# Verified working, public, unauthenticated API (tested Sep 2026).
# NOTE: this is NOT "https://empoweredindian.in/mplads" (that's the human
# dashboard website). The actual API lives on a separate subdomain:
API_BASE_URL = "https://api.empoweredindian.in"

INDIAN_STATES_UTS = [
    "Andhra Pradesh", "Arunachal Pradesh", "Assam", "Bihar", "Chhattisgarh", "Goa",
    "Gujarat", "Haryana", "Himachal Pradesh", "Jharkhand", "Karnataka", "Kerala",
    "Madhya Pradesh", "Maharashtra", "Manipur", "Meghalaya", "Mizoram", "Nagaland",
    "Odisha", "Punjab", "Rajasthan", "Sikkim", "Tamil Nadu", "Telangana", "Tripura",
    "Uttar Pradesh", "Uttarakhand", "West Bengal", "Andaman And Nicobar Islands",
    "Chandigarh", "The Dadra And Nagar Haveli And Daman And Diu", "Delhi",
    "Jammu And Kashmir", "Ladakh", "Lakshadweep", "Puducherry",
]


def fetch_completed_works_live(page_size: int = 200, pause: float = 0.3) -> pd.DataFrame:
    """Pulls every completed work nationwide by paging through each state.
    The API returns camelCase / snake_case fields; we rename them here so
    the rest of the notebook (Sections 3-10) doesn't need to change at all."""
    all_rows = []
    for state in INDIAN_STATES_UTS:
        page = 1
        while True:
            resp = requests.get(
                f"{API_BASE_URL}/api/works/completed",
                params={"state": state, "page": page, "limit": page_size},
                timeout=30,
            )
            resp.raise_for_status()
            payload = resp.json()["data"]
            works = payload.get("completedWorks", [])
            if not works:
                break
            all_rows.extend(works)
            if not payload.get("pagination", {}).get("hasNext"):
                break
            page += 1
            time.sleep(pause)  # be polite to the public API

    raw = pd.DataFrame(all_rows)
    if raw.empty:
        return raw

    comp_live = pd.DataFrame({
        "Work ID": raw["work_id"],
        "Work Description": raw["work_description"],
        "Category": raw["category"],
        "MP Name": raw["mp_details"].apply(lambda d: d.get("name") if isinstance(d, dict) else None),
        "Constituency": raw["mp_details"].apply(lambda d: d.get("constituency") if isinstance(d, dict) else None),
        "State": raw["state"],
        "House": raw["mp_details"].apply(lambda d: d.get("party") if isinstance(d, dict) else None),
        "Final Amount (₹)": raw["cost"],
        "Completed Date": raw["completion_date"],
        # This API does not expose a photo-evidence flag the way your original
        # CSV did, so the "missing evidence" risk signal can't be computed on
        # live data yet. Everything defaults to True (not flagged) until a
        # source for this field is found.
        "Has Images": True,
        "IDA": raw["location"],
    })
    return comp_live


def fetch_mp_summaries_live() -> pd.DataFrame:
    resp = requests.get(
        f"{API_BASE_URL}/api/summary/mps", params={"page": 1, "limit": 800}, timeout=30
    )
    resp.raise_for_status()
    raw = pd.DataFrame(resp.json()["data"])

    mps_live = pd.DataFrame({
        "MP Name": raw["mpName"],
        "Constituency": raw["constituency"],
        "State": raw["state"],
        "House": raw["house"],
        "Allocated Amount (₹)": raw["allocatedAmount"],
        "Total Expenditure (₹)": raw["totalExpenditure"],
        "Utilization %": raw["utilizationPercentage"],
        "Completion Rate %": raw["completionRate"],
        "Completed Works": raw["completedWorksCount"],
        "Recommended Works": raw["recommendedWorksCount"],
        "Transaction Count": raw["completedWorksCount"],  # closest available proxy
        "Balance Not Yet Paid to Vendors (₹)": raw["unpaidBalance"],
    })
    return mps_live


def fetch_live_mplads_data(api_base_url: str = API_BASE_URL, api_key: str = None):
    """
    Live version, built against the real public Empowered Indian API.

    This public API only covers TWO of your four datasets:
      ✅ completed works   -> fetch_completed_works_live()
      ✅ MP summaries      -> fetch_mp_summaries_live()
      ❌ recommended works -> no public per-item endpoint found
      ❌ expenditures      -> no public vendor-level endpoint found

    For the two it doesn't cover, this falls back to whatever you already
    loaded from static CSV in Section 1 (`rec`, `exp`). Replace that
    fallback the moment you find/confirm a live source for those.
    """
    completed_df = fetch_completed_works_live()
    mp_summary_df = fetch_mp_summaries_live()

    recommended_df = rec.copy()   # static CSV fallback (Section 1)
    expenditure_df = exp.copy()   # static CSV fallback (Section 1)

    return completed_df, recommended_df, expenditure_df, mp_summary_df


# --- Switch used at the top of the notebook ---
USE_LIVE_API = False   # flip to True to pull live completed-works + MP data

if USE_LIVE_API:
    comp, rec, exp, mps = fetch_live_mplads_data()
    print(f"Loaded {len(comp):,} live completed works and {len(mps):,} live MP summaries ✅")
    print("Recommended works and expenditures are still the static CSV versions.")
    print("Now re-run Sections 3 through 10.")
else:
    print("Still in static-CSV mode. Set USE_LIVE_API = True once you're ready to pull live data.")


Still in static-CSV mode. Set USE_LIVE_API = True once you're ready to pull live data.
